In [9]:
# Define variables and parameters

import numpy as np
import matplotlib.pyplot as plt

#Input parameters: 
D = 3 * 10**-10         #Diffusion Coefficient m2 s-1
k = 10**-3              #Reaction rate s-1
L = 10**-3              #Length of the domain m

deltau = 1              #Concentration difference across the domain mol m-3
q = -D * deltau / L     #flux at the lower Neumann boundary using Ficks Law mol m-2 s-1
#q = -3 * 10**-7        #flux at the lower Neumann boundary mol m-2 s-1


#The length of the domain in x and y direction
Lx = L
Ly = L

#Neumann Boundary condition values
nbc_lower = q
nbc_upper = 0

# Initial conditions
def initial_condition(N):
    u_init = np.zeros(N)
    return u_init

In [10]:
# Build matrix A

def build_A(Nx, Ny):
    #create grid points in x and y direction
    x = np.linspace(0, Lx, Nx)
    y = np.linspace(0, Ly, Ny)
    
    #calculate grid spacing (delta x and delta y) 
    hx = x[1] - x[0]
    hy = y[1] - y[0]

    # Flattened system size
    N = Nx * Ny

    # Define r
    dt = 1.0  # time step (not used in steady-state Laplace, but kept for consistency)
    rx = ( D * dt) / (hx**2)
    ry = ( D * dt) / (hy**2)
    r = rx # use this for rx = ry

    # ---------------------------
    # 2. Allocate A and b
    # ---------------------------
    A = np.zeros((N, N))

    # Helper: convert (i,j) → k (to be able to solve the system of equations with the correct indexing)
    def idx(i, j):
        return i + j * Nx

    # ---------------------------
    # 3. Fill A and b
    # ---------------------------
    for j in range(Ny):
     for i in range(Nx):
        k = idx(i, j)
        
        # 1) left boundary → Neumann
        if i == 0:
            # (u_1,j - u_0,j)/hx = q
            A[k, idx(1, j)] =  1.0
            A[k, idx(0, j)] = -1.0 

        # 2) right boundary → Neumann
        elif i == Nx-1:
            # (u_Nx,j - u_{Nx-1},j)/hx = q
            A[k, idx(Nx-2, j)] = -1.0
            A[k, idx(Nx-1, j)] =  1.0

        # 3) top → Neumann
        elif j == Ny-1:
            A[k, idx(i, Ny-2)] = -1.0
            A[k, idx(i, Ny-1)] =  1.0

        # 4) bottom → Neumann
        elif j == 0:
            A[k, idx(i, 1)] =  1.0
            A[k, idx(i, 0)] = -1.0

        # 5) interior
        else:
            A[k, k]              = 1 + 4 * r + k * dt
            A[k, idx(i+1, j)]    =  -r
            A[k, idx(i-1, j)]    =  -r
            A[k, idx(i, j+1)]    =  -r
            A[k, idx(i, j-1)]    =  -r

    return A

In [11]:
# build right-hand side b

def build_b(Nx, Ny, u_prev):
    #create grid points in x and y direction
    x = np.linspace(0, Lx, Nx)
    y = np.linspace(0, Ly, Ny)
    
    #calculate grid spacing (delta x and delta y) 
    hx = x[1] - x[0]
    hy = y[1] - y[0]

    # Flattened system size
    N = Nx * Ny

    # ---------------------------
    # 2. Allocate A and b
    # ---------------------------
    b = np.zeros(N)

    # Helper: convert (i,j) → k (to be able to solve the system of equations with the correct indexing)
    def idx(i, j):
        return i + j * Nx

    # ---------------------------
    # 3. Fill A and b
    # ---------------------------
    for j in range(Ny):
     for i in range(Nx):
        k = idx(i, j)
        
        # 1) left boundary → Neumann
        if i == 0:
            # (u_1,j - u_0,j)/hx = q
            b[k]            = nbc_lower*hx  # or q_N(y[j])

        # 2) right boundary → Neumann
        elif i == Nx-1:
            # (u_Nx,j - u_{Nx-1},j)/hx = q
            b[k]               = nbc_upper*hx  # or q_N(y[j])

        # 3) top → Neumann
        elif j == Ny-1:
            b[k]    = nbc_upper*hy  # or q_N(x[i], y[j])
        # 4) bottom → Neumann
        elif j == 0:
            b[k]    = nbc_lower*hy  # or q_N(x[i], y[j])

        # 5) interior
        else:
            b[k] = u_prev[k]

    return b

In [12]:
# solve the PDE using Finite differences

def finite_differences(Nx, Ny, Nt):
    # Build coefficient matrix A
    A = build_A(Nx, Ny)
    
    # Initial condition: u = 0 everywhere
    N = Nx * Ny
    u = initial_condition(N)

    # Time-stepping loop
    for n in range(Nt):
        # Build right-hand side vector b
        b = build_b(Nx, Ny, u)

        # Solve the linear system A u_new = b
        u_new = np.linalg.solve(A, b)

        # Update for next time step
        u = u_new.copy()

    return u_new

In [ ]:
# Grid refinement DOES NOT WORK

def grid_refinement(soln_coarse, Nx_coarse, Ny_coarse, refinement_factor):
    Nx_fine = Nx_coarse * refinement_factor
    Ny_fine = Ny_coarse * refinement_factor
    soln_fine = np.zeros((Nx_fine, Ny_fine))

    for j in range(Ny_coarse):
        for i in range(Nx_coarse):
            soln_fine[i*refinement_factor:(i+1)*refinement_factor,
                       j*refinement_factor:(j+1)*refinement_factor] = soln_coarse[i, j]

    return soln_fine

def plot_refinement(soln_coarse, Nx_coarse, Ny_coarse, refinement_factor):
    soln_fine = grid_refinement(soln_coarse.reshape((Nx_coarse, Ny_coarse)), Nx_coarse, Ny_coarse, refinement_factor)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.title("Coarse Grid Solution")
    plt.imshow(soln_coarse.reshape((Nx_coarse, Ny_coarse)), origin='lower', extent=[0, Lx, 0, Ly], cmap='viridis')
    plt.colorbar(label='Concentration')
    plt.xlabel('x (m)')
    plt.ylabel('y (m)')

    plt.subplot(1, 2, 2)
    plt.title("Refined Grid Solution")
    plt.imshow(soln_fine, origin='lower', extent=[0, Lx, 0, Ly], cmap='viridis')
    plt.colorbar(label='Concentration')
    plt.xlabel('x (m)')
    plt.ylabel('y (m)')

    plt.tight_layout()
    plt.show()